# SWIG counterfactual examples

This notebook exercises `VibecodeSwig` and `SwigCounterfactualEngine` on small classical discrete examples.

Semantic note: CPDs alone do not uniquely determine all counterfactuals. The engine uses the canonical discrete response-table SCM described in `vibecode_swig.py`: each CPD column induces an independent latent response-table entry.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

from pgmpy.factors.discrete import TabularCPD
from pgmpy.models import DiscreteBayesianNetwork

from reasoning_core.tasks.vibecode_swig import VibecodeSwig, SwigCounterfactualEngine

In [ ]:
def make_bn(edges, cpds):
    bn = DiscreteBayesianNetwork(edges)
    bn.add_cpds(*cpds)
    assert bn.check_model()
    return bn


def run_case(name, bn, interventions, target, factual_evidence, n_round=4):
    swig = VibecodeSwig.from_bn(bn, interventions)
    swig.validate_swig_metadata()
    assert swig.check_model()
    engine = SwigCounterfactualEngine(swig, max_response_functions=2_000_000)
    result = engine.query(target, factual_evidence, n_round=n_round)
    assert abs(sum(result.values()) - 1.0) < 1e-8
    print(f"\n=== {name} ===")
    print("Interventions:", interventions)
    print("Factual evidence:", factual_evidence)
    print("Target SWIG node:", swig.random_node_for(target))
    print("Counterfactual distribution:", result)
    print("P(evidence):", round(engine.last_evidence_probability, n_round))
    print("Response-space size:", engine.response_space_size())
    print("Trace:")
    print("\n".join(engine.trace))
    return swig, engine, result

## 1. Deterministic switch

`Y = X`. Given the factual world `X=0, Y=0`, under `do(X=1)` the counterfactual `Y` should be 1 with probability 1.

In [ ]:
cpd_x = TabularCPD("X", 2, [[0.5], [0.5]], state_names={"X": [0, 1]})
cpd_y_eq_x = TabularCPD(
    "Y", 2,
    [[1.0, 0.0], [0.0, 1.0]],
    evidence=["X"], evidence_card=[2],
    state_names={"Y": [0, 1], "X": [0, 1]},
)
bn_switch = make_bn([("X", "Y")], [cpd_x, cpd_y_eq_x])

run_case(
    "deterministic switch",
    bn_switch,
    interventions={"X": 1},
    target="Y",
    factual_evidence={"X": 0, "Y": 0},
)

## 2. Probabilistic treatment

`P(Y=1 | X=0)=0.1` and `P(Y=1 | X=1)=0.8`. Under the canonical response-table semantics, observing `Y=0` at `X=0` does not constrain the response-table entry for `X=1`, so `P(Y_{X=1}=1 | X=0,Y=0)=0.8`.

In [ ]:
cpd_x = TabularCPD("X", 2, [[0.5], [0.5]], state_names={"X": [0, 1]})
cpd_y_treatment = TabularCPD(
    "Y", 2,
    [[0.9, 0.2], [0.1, 0.8]],
    evidence=["X"], evidence_card=[2],
    state_names={"Y": [0, 1], "X": [0, 1]},
)
bn_treatment = make_bn([("X", "Y")], [cpd_x, cpd_y_treatment])

run_case(
    "probabilistic treatment",
    bn_treatment,
    interventions={"X": 1},
    target="Y",
    factual_evidence={"X": 0, "Y": 0},
)

## 3. Confounded treatment

`U` affects both treatment `X` and outcome `Y`. Factual evidence about `X` and `Y` updates the posterior over the latent response tables and the observed confounder `U` response, then prediction is made under `do(X=1)`.

In [ ]:
cpd_u = TabularCPD("U", 2, [[0.5], [0.5]], state_names={"U": [0, 1]})
cpd_x_given_u = TabularCPD(
    "X", 2,
    [[0.9, 0.1], [0.1, 0.9]],
    evidence=["U"], evidence_card=[2],
    state_names={"X": [0, 1], "U": [0, 1]},
)
# Parent order for Y is X, U. Columns: (X=0,U=0), (X=0,U=1), (X=1,U=0), (X=1,U=1).
p_y1 = [0.05, 0.70, 0.60, 0.95]
cpd_y_given_xu = TabularCPD(
    "Y", 2,
    [[1 - p for p in p_y1], p_y1],
    evidence=["X", "U"], evidence_card=[2, 2],
    state_names={"Y": [0, 1], "X": [0, 1], "U": [0, 1]},
)
bn_confounding = make_bn(
    [("U", "X"), ("U", "Y"), ("X", "Y")],
    [cpd_u, cpd_x_given_u, cpd_y_given_xu],
)

run_case(
    "confounded treatment",
    bn_confounding,
    interventions={"X": 1},
    target="Y",
    factual_evidence={"X": 0, "Y": 0},
)

## 4. Mediated deterministic effect

`X -> M -> Y`, with `M=X` and `Y=M`. Given `X=0,M=0,Y=0`, under `do(X=1)` the mediated counterfactual outcome is 1.

In [ ]:
cpd_x = TabularCPD("X", 2, [[0.5], [0.5]], state_names={"X": [0, 1]})
cpd_m_eq_x = TabularCPD(
    "M", 2,
    [[1.0, 0.0], [0.0, 1.0]],
    evidence=["X"], evidence_card=[2],
    state_names={"M": [0, 1], "X": [0, 1]},
)
cpd_y_eq_m = TabularCPD(
    "Y", 2,
    [[1.0, 0.0], [0.0, 1.0]],
    evidence=["M"], evidence_card=[2],
    state_names={"Y": [0, 1], "M": [0, 1]},
)
bn_mediation = make_bn([("X", "M"), ("M", "Y")], [cpd_x, cpd_m_eq_x, cpd_y_eq_m])

run_case(
    "mediated deterministic effect",
    bn_mediation,
    interventions={"X": 1},
    target="Y",
    factual_evidence={"X": 0, "M": 0, "Y": 0},
)

## 5. But-for cause with an alternative sufficient cause

`Y = X OR Z`. First we condition on `Z=0`, so removing `X` makes `Y` false. Then we omit `Z` from the factual evidence; because `X=1,Y=1` does not reveal whether `Z=1`, the counterfactual keeps uncertainty about the alternative cause.

In [ ]:
cpd_x = TabularCPD("X", 2, [[0.5], [0.5]], state_names={"X": [0, 1]})
cpd_z = TabularCPD("Z", 2, [[0.5], [0.5]], state_names={"Z": [0, 1]})
# Parent order for Y is X, Z. Y = X OR Z.
cpd_y_or = TabularCPD(
    "Y", 2,
    [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 1.0, 1.0]],
    evidence=["X", "Z"], evidence_card=[2, 2],
    state_names={"Y": [0, 1], "X": [0, 1], "Z": [0, 1]},
)
bn_or = make_bn([("X", "Y"), ("Z", "Y")], [cpd_x, cpd_z, cpd_y_or])

run_case(
    "but-for cause when Z is known absent",
    bn_or,
    interventions={"X": 0},
    target="Y",
    factual_evidence={"X": 1, "Z": 0, "Y": 1},
)

run_case(
    "but-for uncertainty when Z is unobserved",
    bn_or,
    interventions={"X": 0},
    target="Y",
    factual_evidence={"X": 1, "Y": 1},
)

## Inspecting the SWIG object

The SWIG is still a `DiscreteBayesianNetwork`, but the SWIG-specific metadata tells us which nodes are fixed intervention values and which are random counterfactual variables.

In [ ]:
swig = VibecodeSwig.from_bn(bn_or, {"X": 0})
print(swig.to_nl())
print("\nCounterfactual nodes:", swig.counterfactual_nodes())
print("\nSerializable metadata keys:", swig.to_serializable().keys())